# Week 4: Transparent Rule Baseline & Hand Review (ML-07)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  
**Primary Metric:** Precision@50 (vs 54.2% Base Rate)  

---

## 1. The Principle of an Honest Baseline

In accordance with `skills/building-baselines/SKILL.md`: *"A model without a baseline is a number without a meaning."*

Before fitting any machine-learned model, we build a transparent, readable heuristic score that reflects standard SEO industry best practices. Its purpose is to be an honest, frozen benchmark that any complex model must tangibly outperform.

## 2. Baseline Heuristic Scoring Formula

We construct a 4-component weighted composite score bounded in $[0, 1]$:

$$\text{Baseline Score} = 0.40 \cdot S_{\text{vis}} + 0.30 \cdot S_{\text{fresh}} + 0.25 \cdot S_{\text{pos}} + 0.05 \cdot S_{\text{depth}}$$

### 2.1 Score Component Definitions
1. **Visibility Opportunity ($S_{\text{vis}}$, weight = 0.40):**
   $$S_{\text{vis}} = \text{PercentileRank}(\log(1 + \text{impressions\_90d}))$$
   Measures total available organic search demand in Google.

2. **Freshness Risk ($S_{\text{fresh}}$, weight = 0.30):**
   $$S_{\text{fresh}} = \text{PercentileRank}(\text{days\_since\_last\_update})$$
   Penalizes content that has not undergone editorial updates.

3. **Position Opportunity ($S_{\text{pos}}$, weight = 0.25):**
   $$S_{\text{pos}} = \left(1 - \frac{\text{clip}(\text{avg\_position}, 1, 50) - 1}{49}\right) \cdot S_{\text{vis}} \cdot \mathbb{I}_{\text{avg\_position} > 0}$$
   High value is assigned to pages ranking near the top of SERPs (Pages 1 and 2).

4. **Depth Gap ($S_{\text{depth}}$, weight = 0.05):**
   $$S_{\text{depth}} = (1 - \text{PercentileRank}(\text{word\_count})) \cdot S_{\text{vis}}$$
   Modest penalty for thin content receiving organic search impressions.

## 3. Reason Code Heuristic Logic

Every scored page carries transparent diagnostic reason codes explaining *why* it was flagged:

```python
def evaluate_baseline_reasons(row):
    reasons = []
    if row["days_since_last_update"] >= 180 and row["impressions_90d"] >= 500:
        reasons.append("stale_visible_page")
    if row["trend_direction"].lower() == "down" and row["impressions_90d"] >= 100:
        reasons.append("declining_with_demand")
    if 0 < row["word_count"] < 1200 and row["impressions_90d"] >= 250:
        reasons.append("thin_visible_page")
    if 0 < row["avg_position"] <= 10 and row["content_age_days"] >= 180:
        reasons.append("page_one_decay_risk")
    if row["impressions_90d"] >= 500 and 0 < row["avg_position"] <= 20 and row["ctr"] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row["sessions_90d"] >= 30 and (row["engagement_rate"] < 30 or row["scroll_rate"] < 30):
        reasons.append("low_engagement_visible_page")
    return reasons or ["general_refresh_review"]
```

## 4. Empirical Baseline Evaluation

Evaluating the deterministic baseline across the 30,000-row dataset under client-holdout validation yields the following benchmark numbers:

| Evaluation Metric | Baseline Value | Majority-Class Base Rate | Interpretation |
|---|---:|---:|---|
| **Precision@50** | **0.240** | **0.542** | **Underperforms random guessing.** Only 12 of the top 50 pages are actually declining. |
| **ROC AUC** | **0.627** | 0.500 | Modest global separation across all 30k rows. |
| **Average Precision** | **0.468** | 0.542 | PR curve underperforms baseline class balance. |

### 4.1 The Top-20 Hand Review: Why Does the Heuristic Fail?
A manual inspection of the top 20 items ranked by the baseline score reveals the classic failure mode of fixed SEO heuristics:
1. **The "Giant Evergreen" Trap:** The visibility term ($S_{\text{vis}}$) and staleness term ($S_{\text{fresh}}$) heavily reward huge, authoritative guide articles published 2 years ago that receive 50,000 impressions.
2. **False Urgency:** Many of these massive evergreen guides are **completely stable** or even growing. The heuristic flags them as "urgent" solely because of high volume and elapsed calendar time.
3. **Opportunity Cost:** While editors waste hours inspecting massive healthy pages, subtle early-stage decay on high-intent conversion articles is completely buried down at ranks #500–#2,000.

## 5. Rubric Pass Checklist (ML-07)

- [x] **Transparent rule formulated:** Formula stated clearly with 4 normalized components.
- [x] **Reason codes attached:** 6 diagnostic tags explain why an item is scored.
- [x] **Precision@50 computed:** Baseline achieves 0.240 (establishing the benchmark floor).
- [x] **Base rate reported:** 54.2% base rate reported alongside precision.
- [x] **Top-20 hand review documented:** Identified the "giant evergreen guide" false positive pattern.
- [x] **Baseline frozen:** Kept immutable as the benchmark against which machine learning models are evaluated in Week 5.